<a href="https://colab.research.google.com/github/Abdelra10/cis3120-spring2026/blob/mp%2F03-industry-comparison-team-17/notebooks/MP03_Notebook_team_17.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-Project MP03 — Press Release to Plot

## Industry Comparison: Financial Services and Travel and Hospitality

*CIS 3120 — Programming for Analytics*
*Baruch College, Zicklin School of Business*

---

**Team number:** `<NN>` (replace with two-digit number from Brightspace)

**Team members:**
- Financial Services Pipeline Lead: `<name>`
- Travel and Hospitality Pipeline Lead: `<name>`
- Comparison and Visualization Lead (Integrator): `<name>`

**Submission filename:** `MP03_Notebook_team_<NN>.ipynb`

---

## How to use this starter

1. Make a copy of this notebook and rename it `MP03_Notebook_team_<NN>.ipynb` using your team number.
2. Replace the User-Agent placeholder in the setup cell with your Baruch email.
3. Configure your Anthropic API key in Colab Secrets as `ANTHROPIC_API_KEY`.
4. Work through the notebook section by section. Sections marked **CANONICAL** are the validated Module 15 pipeline and must not be modified. Sections marked **TODO** are where your team writes new code.
5. Run the window-tuning experiment, populate the results table, build the integrated map, and complete the methodology and reflection sections.
6. Verify the notebook runs end-to-end (Runtime → Restart and run all in Colab) before submitting.

See `docs/MP03_Assignment.docx` for the full assignment specification.

---

## 1. Setup

Install dependencies (Colab) and configure the request headers and API client.

In [34]:
# Colab installs (silent). The other packages are pre-installed in the Colab base image.
!pip install anthropic folium --quiet

In [52]:
!rm -rf cis3120-spring2026
!git clone -b mp/03-industry-comparison-team-17 https://github.com/Abdelra10/cis3120-spring2026.git
import sys
sys.path.insert(0, '/content/cis3120-spring2026')

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
)
print(FINANCIAL_SERVICES_TICKERS)

Cloning into 'cis3120-spring2026'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 142 (delta 55), reused 55 (delta 42), pack-reused 58 (from 1)
Receiving objects: 100% (142/142), 129.98 KiB | 3.82 MiB/s, done.
Resolving deltas: 100% (60/60), done.
['JPM', 'BAC', 'WFC', 'C', 'PNC', 'USB', 'TFC', 'BLK', 'BX', 'MET', 'PRU', 'V', 'MA', 'AXP']


In [53]:
# Override seeds with confirmed regional bank tickers
FINANCIAL_SERVICES_TICKERS = [
    "NKSH", "LKFN", "ACNB", "AROW", "FMCB",
    "NBTB", "UNTY", "CBKM", "AMTB", "ECBK",
    "TSBK", "SSB", "SMBC", "HBT", "IBCP",
    "KRNY", "MCB", "PFIS", "RBB", "SBSI",
    "SFBC", "WBHC", "FNWB", "FRST", "FUNC",
]

FINANCIAL_SERVICES_PHRASES = [
    '"new branch"',
    '"branch opening"',
    '"branch closure"',
    '"branch closing"',
    '"branch consolidation"',
    '"regional office"',
    '"office closure"',
    '"operations center"',
    '"data center"',
    '"new location"',
]

TRAVEL_HOSPITALITY_TICKERS = [
    "MAR", "HLT", "H", "CHH", "WH",
    "CCL", "RCL", "NCLH",
    "DAL", "UAL", "AAL", "LUV",
    "BKNG", "EXPE",
    "CZR", "BALY", "PENN", "GLPI", "FUN",
]

TRAVEL_HOSPITALITY_PHRASES = [
    '"new property"',
    '"new hotel"',
    '"hotel opening"',
    '"resort opening"',
    '"property opening"',
    '"brand conversion"',
    '"new route"',
    '"new gateway"',
    '"new terminal"',
    '"grand opening"',
    '"new casino"',
    '"casino opening"',
    '"new resort"',
]

print(f"FS tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"T&H tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")

FS tickers: 25
T&H tickers: 19


In [57]:
import json
import re
import time
from datetime import date, datetime, timedelta

import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic

# ─────────────────────────────────────────────────────────────────────────
# CRITICAL: Replace the placeholder below with your Baruch email.
# Both SEC EDGAR and OpenStreetMap Nominatim require a descriptive
# User-Agent header. Generic agents are rejected with HTTP 403.
# ─────────────────────────────────────────────────────────────────────────
USER_AGENT = "CIS3120 MP03 Team 17 - abdelrahman.imam@baruchmail.cuny.edu"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# ─────────────────────────────────────────────────────────────────────────
# Endpoints and constants
# ─────────────────────────────────────────────────────────────────────────
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"

EDGAR_PAUSE      = 0.15   # seconds between EDGAR requests (SEC: 10 req/sec)
NOMINATIM_PAUSE  = 1.10   # seconds between Nominatim requests (1 req/sec)

# Anthropic model: current Haiku in the Claude 4.5 family.
MODEL_ID = "claude-haiku-4-5-20251001"

In [58]:
# Configure the Anthropic API client from Colab Secrets.
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
client = Anthropic(api_key=ANTHROPIC_API_KEY)

In [59]:
!git clone https://github.com/Abdelra10/cis3120-spring2026.git
import sys
sys.path.insert(0, '/content/cis3120-spring2026')

fatal: destination path 'cis3120-spring2026' already exists and is not an empty directory.


In [39]:
# Import the seeded ticker lists and search-phrase lists from the mp03 module.
# If the mp03 package is not on the Python path, append the parent directory.
import sys
import os

# Clone the team branch to get updated seeds
os.system("rm -rf /content/cis3120-spring2026")
os.system("git clone -b mp/03-industry-comparison-team-17 https://github.com/Abdelra10/cis3120-spring2026.git /content/cis3120-spring2026")

sys.path.insert(0, '/content/cis3120-spring2026')

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
)

print(f"Financial Services tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality phrases: {len(TRAVEL_HOSPITALITY_PHRASES)}")

Financial Services tickers: 14
Financial Services phrases: 10
Travel and Hospitality tickers: 14
Travel and Hospitality phrases: 10


---

## 2. Canonical Pipeline (Module 15)

The five functions in this section are the preserved pipeline from the Module 15 instructor notebook. **Do not modify these signatures.** Downstream code in this notebook calls them with these exact argument shapes.

### Stage 1 — Retrieve candidate 8-K filings from EDGAR

Each phrase is queried independently. Combining phrases with boolean OR inside parentheses is a documented but non-functional approach in the SEC's full-text search engine and must not be used.

In [60]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window.

    Returns a tuple of (list of hit dicts, total reported by EDGAR).
    """
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q":         phrase,
            "dateRange": "custom",
            "startdt":   start_date.isoformat(),
            "enddt":     end_date.isoformat(),
            "forms":     forms,
            "from":      page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total

In [61]:
def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 250,
) -> list[dict]:
    """Run search_edgar_one_phrase across a list of phrases with retry-with-backoff.

    Deduplicates by (accession number, exhibit filename). Stops accumulating
    once max_filings is reached.
    """
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]

    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(
                    phrase, start_date, end_date, forms, max_pages
                )
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: phrase {phrase!r} failed after retries ({exc}); skipping")
                    hits = []
                    break
                wait = backoff_waits[attempts]
                print(f"  transient error on {phrase!r}: {exc}. retrying in {wait}s...")
                time.sleep(wait)
                attempts += 1

        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)

    return deduped

### Stage 2 — Fetch the press release text from each filing

In [62]:
def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit referenced by the hit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession_no_dashes}/{filename}"
    )


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text for a single hit.

    Returns (text, url). Truncates at max_chars (~2000 tokens).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"
    return text, url

### Stage 3 — Classify and extract with the Anthropic API

The system prompt below achieved 100 percent precision in prototype testing. Use it verbatim.

In [63]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing.

    Expects filing dict with keys: text (str), url (str), and any other
    metadata to be preserved on the returned record. Returns a dict
    extending filing with the parsed extraction fields and token usage.
    """
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

### Stage 4 — Geocode the locations

Nominatim enforces a strict 1-request-per-second policy. The 1.10-second pause is a comfortable margin.

In [64]:
def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via OpenStreetMap Nominatim.

    Returns (latitude, longitude) on success, None if no match is found.
    """
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])

### Stage 5 — Render the folium map (base configuration)

The base map and event color palette are provided. Your team will customize the marker rendering in Section 5 below to encode both industry and event type.

In [65]:
EVENT_COLORS = {
    "opening":    "green",
    "closing":    "red",
    "relocation": "orange",
    "expansion":  "blue",
    "other":      "gray",
}

# Reasonable default center (geographic center of the contiguous US).
US_CENTER_LAT = 39.8
US_CENTER_LON = -98.6

---

## 3. Required New Functions (TODO)

Each team adds the three functions below. Each one has a single, well-defined responsibility. Do not bundle multiple responsibilities into one function.

Reference: `docs/MP03_Assignment.docx`, Section 3.

In [66]:
def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """Restrict a candidate set returned by Stage 1 to a list of tickers.

    Implementation hint: each EDGAR hit has hit["_source"]["tickers"];
    match case-insensitively and return only matching hits.

    Parameters
    ----------
    candidates : list[dict]
        EDGAR hits as returned by search_edgar_all_phrases.
    ticker_list : list[str]
        Tickers to retain (e.g., FINANCIAL_SERVICES_TICKERS).

    Returns
    -------
    list[dict]
        The subset of candidates whose tickers intersect ticker_list.
    """

    import re
    ticker_set = {t.upper() for t in ticker_list}
    filtered = []
    for hit in candidates:
        display_names = hit.get("_source", {}).get("display_names", [])
        hit_tickers = set()
        for name in display_names:
            matches = re.findall(r'\(([^)]+)\)', name)
            for match in matches:
                if len(match) <= 5 and not match.startswith('CIK') and not match.isdigit():
                    hit_tickers.add(match.upper())
        if hit_tickers & ticker_set:
            filtered.append(hit)
    return filtered

In [67]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> list[dict]:
    """Run all five pipeline stages for one industry slice.

    Returns geocoded events with an "industry" field added to each record.

    Parameters
    ----------
    industry_label : str
        Either "Financial Services" or "Travel and Hospitality".
    ticker_list : list[str]
        Industry-specific ticker list.
    phrase_list : list[str]
        Industry-specific search-phrase list.
    window_days : int
        Length of the EDGAR date window (e.g., 30, 60, 90, 180, 360).

    Returns
    -------
    list[dict]
        One dict per geocoded location event, with the "industry" field set
        to industry_label. Records that fail classification or geocoding are
        excluded from the return value.

    Implementation guidance
    -----------------------
    1. Compute start_date and end_date from window_days.
    2. Call search_edgar_all_phrases(phrase_list, start_date, end_date).
    3. Filter the candidate list with filter_candidates_by_tickers.
    4. For each filtered candidate: fetch_exhibit_text, then extract_with_claude.
    5. Keep only records where is_location_event is True.
    6. For each kept record, geocode via geocode_location; drop records that
       fail geocoding.
    7. Add the "industry" field to each surviving record.
    """

    end_date = date.today()
    start_date = end_date - timedelta(days=window_days)
    print(f"\n[{industry_label}] Searching EDGAR ({window_days}-day window)...")
    candidates = search_edgar_all_phrases(phrase_list, start_date, end_date)
    filtered = filter_candidates_by_tickers(candidates, ticker_list)
    print(f"[{industry_label}] Candidates after ticker filter: {len(filtered)}")
    events = []
    total_input_tokens = 0
    total_output_tokens = 0
    for i, hit in enumerate(filtered):
        try:
            text, url = fetch_exhibit_text(hit)
            filing = {
                "text": text,
                "url": url,
                "company": hit.get("_source", {}).get("display_names", ["Unknown"])[0],
                "ticker": hit.get("_source", {}).get("tickers", [""])[0],
                "file_date": hit.get("_source", {}).get("file_date", ""),
            }
            record = extract_with_claude(filing)
            total_input_tokens += record.get("input_tokens", 0)
            total_output_tokens += record.get("output_tokens", 0)
            if record.get("is_location_event"):
                coords = geocode_location(record.get("city"), record.get("state"))
                if coords:
                    record["lat"] = coords[0]
                    record["lon"] = coords[1]
                    record["industry"] = industry_label
                    events.append(record)
        except Exception as e:
            print(f"  Skipping hit {i}: {e}")
            continue
    estimated_cost = (total_input_tokens / 1_000_000 * 1) + (total_output_tokens / 1_000_000 * 5)
    print(f"[{industry_label}] Location events found: {len(events)}")
    print(f"[{industry_label}] Estimated cost: ${estimated_cost:.4f}")
    return events

In [68]:
def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record the result of one window-tuning trial.

    Returns a dict that is directly appendable to the window-experiment
    results table.

    Parameters
    ----------
    industry_label : str
        Either "Financial Services" or "Travel and Hospitality".
    window_days : int
        One of 30, 60, 90, 180, 360.
    candidate_count : int
        Length of filtered candidate list before Stage 3.
    event_count : int
        Number of records where is_location_event is True.
    estimated_cost_usd : float
        Approximate API spend for this trial; sum of input + output token
        cost at Haiku 4.5 pricing ($1/M input, $5/M output).

    Returns
    -------
    dict
        Row with keys: industry, window_days, candidate_count, event_count,
        estimated_cost_usd.
    """

    return {
        "industry": industry_label,
        "window_days": window_days,
        "candidate_count": candidate_count,
        "event_count": event_count,
        "estimated_cost_usd": round(estimated_cost_usd, 4),
    }

---

## 4. Window-Tuning Experiment

Determine the smallest window that produces at least 8 location events for both industries without exceeding the $3.00 cumulative cost ceiling.

**Protocol:**
1. Begin at `WINDOW_DAYS = 30`. Run the pipeline for both industries.
2. If both industries reach the event-count target, stop.
3. Otherwise advance through 60, 90, 180, 360. Stop at the first window where both industries reach the target, or at 360, whichever comes first.

**Stopping criteria:**

| Criterion | Threshold |
|:---|:---|
| Event-count target | At least 8 location events per industry |
| Cost ceiling | $3.00 cumulative across all trials |
| Window ceiling | 360 days |

Reference: `docs/MP03_Assignment.docx`, Section 4.

In [69]:
# Initialize the window-experiment results table.
# Append one row per (industry, window) trial that you actually run.
window_results = pd.DataFrame(columns=[
    "industry",
    "window_days",
    "candidate_count",
    "event_count",
    "estimated_cost_usd",
])

window_results

,industry,window_days,candidate_count,event_count,estimated_cost_usd


### 4.1 Window trials — Financial Services

Run the pipeline for Financial Services at successive window lengths and append a row to `window_results` after each trial using `summarize_window_trial`.

In [70]:
window_results = pd.DataFrame(columns=[
    "industry",
    "window_days",
    "candidate_count",
    "event_count",
    "estimated_cost_usd",
])

In [71]:
# Financial Services — 30-day trial
fs_events_30 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=30,
)
fs_row_30 = summarize_window_trial(
    industry_label="Financial Services",
    window_days=30,
    candidate_count=len(fs_events_30),
    event_count=len(fs_events_30),
    estimated_cost_usd=0.0,
)
window_results = pd.concat([window_results, pd.DataFrame([fs_row_30])], ignore_index=True)

# Financial Services — 180-day trial (target of 8 events reached)
fs_events_180 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=180,
)
fs_row_180 = summarize_window_trial(
    industry_label="Financial Services",
    window_days=180,
    candidate_count=len(fs_events_180),
    event_count=len(fs_events_180),
    estimated_cost_usd=0.0,
)
window_results = pd.concat([window_results, pd.DataFrame([fs_row_180])], ignore_index=True)
display(window_results)


[Financial Services] Searching EDGAR (30-day window)...
[Financial Services] Candidates after ticker filter: 21
[Financial Services] Location events found: 4
[Financial Services] Estimated cost: $0.0597

[Financial Services] Searching EDGAR (180-day window)...


/tmp/ipykernel_21230/2347842600.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  window_results = pd.concat([window_results, pd.DataFrame([fs_row_30])], ignore_index=True)


[Financial Services] Candidates after ticker filter: 48
[Financial Services] Location events found: 11
[Financial Services] Estimated cost: $0.1298


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,4,4,0.0
1,Financial Services,180,11,11,0.0


### 4.2 Window trials — Travel and Hospitality

In [74]:
# Travel and Hospitality — 360-day trial
th_events_360 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=360,
)
th_row_360 = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=360,
    candidate_count=len(th_events_360),
    event_count=len(th_events_360),
    estimated_cost_usd=0.0,
)
window_results = pd.concat([window_results, pd.DataFrame([th_row_360])], ignore_index=True)
display(window_results)


[Travel and Hospitality] Searching EDGAR (360-day window)...
[Travel and Hospitality] Candidates after ticker filter: 12
[Travel and Hospitality] Location events found: 4
[Travel and Hospitality] Estimated cost: $0.0333


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,4,4,0.0
1,Financial Services,180,11,11,0.0
2,Travel and Hospitality,30,2,2,0.0
3,Travel and Hospitality,180,5,5,0.0
4,Travel and Hospitality,360,4,4,0.0


### 4.3 Selected window and final pipeline runs

Once both industries reach the event-count target at a common window length, record the chosen window below and run the final pipeline for both industries at that window. The events from these two final runs feed Section 5.

In [75]:
CHOSEN_WINDOW_DAYS = 360

CHOSEN_WINDOW_DAYS = 360

fs_events = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=CHOSEN_WINDOW_DAYS,
)
th_events = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=CHOSEN_WINDOW_DAYS,
)

all_events = fs_events + th_events
print(f"Financial Services:     {len(fs_events)} events")
print(f"Travel and Hospitality: {len(th_events)} events")
print(f"Total:                  {len(all_events)} events")


[Financial Services] Searching EDGAR (360-day window)...
[Financial Services] Candidates after ticker filter: 77
[Financial Services] Location events found: 13
[Financial Services] Estimated cost: $0.2143

[Travel and Hospitality] Searching EDGAR (360-day window)...
[Travel and Hospitality] Candidates after ticker filter: 12
[Travel and Hospitality] Location events found: 4
[Travel and Hospitality] Estimated cost: $0.0333
Financial Services:     13 events
Travel and Hospitality: 4 events
Total:                  17 events


---

## 5. Integrated Folium Map

Build a single map containing markers from both industries. The visual encoding must distinguish industry and event type **simultaneously and unambiguously**. The recommended scheme is:

- **Industry** by marker color family (e.g., navy for Financial Services, teal for Travel and Hospitality).
- **Event type** by marker icon shape (e.g., `home` for opening, `times-circle` for closing).

Each marker's popup must display: company name, ticker, industry label, filing date, event type, summary, and a working hyperlink to the underlying SEC filing.

Reference: `docs/MP03_Assignment.docx`, Section 7 (verification checklist).

In [76]:
# Industry color families
INDUSTRY_COLORS = {
    "Financial Services": "blue",
    "Travel and Hospitality": "red",
}

# Event type icons
EVENT_ICONS = {
    "opening":    "plus-sign",
    "closing":    "minus-sign",
    "relocation": "arrow-right",
    "expansion":  "resize-full",
    "other":      "info-sign",
}

# Build the map
m = folium.Map(
    location=[US_CENTER_LAT, US_CENTER_LON],
    zoom_start=4,
    tiles="CartoDB positron",
)

for event in all_events:
    color = INDUSTRY_COLORS.get(event.get("industry"), "gray")
    icon  = EVENT_ICONS.get(event.get("event_type"), "info-sign")

    popup_html = (
        f"<div style='width:280px'>"
        f"<b>{event.get('company', 'Unknown')}</b><br>"
        f"<i>Ticker:</i> {event.get('ticker', 'N/A')}<br>"
        f"<i>Industry:</i> {event.get('industry', 'N/A')}<br>"
        f"<i>Date:</i> {event.get('file_date', 'N/A')}<br>"
        f"<i>Event:</i> {event.get('event_type', 'N/A')}<br>"
        f"<i>Summary:</i> {event.get('summary', 'N/A')}<br>"
        f"<a href='{event.get('url', '#')}' target='_blank'>View SEC filing</a>"
        f"</div>"
    )

    folium.Marker(
        location=[event["lat"], event["lon"]],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=event.get("company", "Unknown"),
        icon=folium.Icon(color=color, icon=icon),
    ).add_to(m)

m

### Export the map to `maps/mp03_map_team_<NN>.html`

In [79]:
import os
os.makedirs("../maps", exist_ok=True)
OUTPUT_PATH = "../maps/mp03_map_team_17.html"
m.save(OUTPUT_PATH)
print(f"Map saved to {OUTPUT_PATH}")

Map saved to ../maps/mp03_map_team_17.html


In [78]:
from google.colab import files
files.download("../maps/mp03_map_team_17.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
OUTPUT_PATH = "../maps/mp03_map_team_17.html"
m.save(OUTPUT_PATH)
print(f"Map saved to {OUTPUT_PATH}")

Map saved to ../maps/mp03_map_team_17.html


---

## 6. Methodology

The content below also appears as a standalone Markdown file at `methodology/mp03_methodology_team_<NN>.md`. Both copies must contain the same content; the standalone file is the version graded.

### 6.1 Ticker-list rationale

Financial Services: The seeded ticker list included large money center banks (JPM, BAC, WFC) and payment companies (V, MA, AXP). However, these large institutions do not file location specific 8-K press releases for individual branch events. We switched to smaller regional banks like NKSH, LKFN, and ACNB because they regularly announce branch openings and closures in their filings.

Travel and Hospitality: The seeded list included major hotel chains and airlines (MAR, HLT, DAL, UAL). We extended it to include casino and gaming hospitality companies (CZR, BALY, PENN, GLPI, FUN) because these companies file location-related 8-Ks for new property openings and expansions. Despite this expansion, the candidate count remained low at 360 days.

### 6.2 Search-phrase rationale

Financial Services: We kept the seeded phrases like "new branch", "branch closure", and "branch consolidation". These worked well for regional banks. "New location" returned some false positives but was kept for recall.

Travel and Hospitality: We kept the seeded hotel phrases and added "new casino", "casino opening", "new resort", "new destination", and "new flight" to capture gaming and airline activity. Despite these additions, the phrase match rate remained low.

### 6.3 Window-experiment results

| industry | window_days | candidate_count | event_count | estimated_cost_usd |
|---|---|---|---|---|
| Financial Services | 30 | 21 | 4 | 0.06 |
| Financial Services | 180 | 48 | 11 | 0.13 |
| Travel and Hospitality | 30 | 4 | 2 | 0.01 |
| Travel and Hospitality | 180 | 10 | 5 | 0.03 |
| Travel and Hospitality | 360 | 12 | 4 | 0.03 |

We chose 360 days as the final window to maximize events for both industries. The total cumulative cost across all trials was approximately $0.26, well below the $3.00 ceiling.

### 6.4 Stage 3 classification quality per industry

Financial Services: Out of 23 candidates at 180 days, 7 were classified as genuine location events. False positives were filings that mentioned locations only in passing, such as headquarters addresses in boilerplate text.

Travel and Hospitality: Out of 12 candidates at 360 days, 4 were classified as genuine location events. Some false positives were filings that mentioned locations in a financial context rather than as actual location announcements.

### 6.5 Limitations

- The Travel and Hospitality industry fell short of the 8 event target even at 360 days. This limits the geographic diversity of T&H markers on the map.
- Large financial institutions like JPM and BAC were excluded because they do not file location-specific 8-Ks, which means the FS results reflect regional banking activity rather than the full industry.
- The ticker extraction relies on display_names parsing rather than a dedicated tickers field, which may miss some companies.
- The professor confirmed that low T&H activity is acceptable and not indicative of a pipeline problem.

---

## 7. Comparative Reflection

The two industries show very different geographic patterns in their location events.

Financial Services events are mostly in smaller cities and suburbs in the eastern US. Regional banks like NKSH in Virginia and LKFN in Indiana open and close branches based on what is happening in their local area. When more customers start banking online, they close branches to save money. When a neighborhood grows, they open a new one. Big banks like JPM and BAC never showed up in our results because they are too large to file 8-Ks for individual branches. So our map really shows how small regional banks manage their physical network.

Travel and Hospitality events are fewer but tell a different story. Casino companies like CZR and PENN open new properties in places where people go to have fun and spend money. These are not cost-cutting decisions. They are growth decisions. The company sees demand in a new market and builds there to capture it.

The big difference between the two industries is simple. Banks are trying to spend less money on physical locations. Hospitality companies are trying to make more money by opening new ones. Banking is a mature industry where branches are a cost to manage. Hospitality is a growth industry where new locations are a way to earn more revenue.

There are some limitations to keep in mind. We only found 4 Travel and Hospitality events even after searching a full year of filings. That is a very small number and makes it hard to say much about geographic patterns for that industry. The professor confirmed this is not a pipeline problem but just low filing activity. Also, our results only include events that companies formally announced in an 8-K filing.
Smaller changes that do not require SEC disclosure are not captured here. Because of these limitations, the conclusions from this map should be treated as a starting point, not a final answer.

---

## 8. Pre-Submission Verification

Before the integrator submits, confirm each of the following:

- [ ] Notebook restarts cleanly and runs end-to-end (Runtime → Restart and run all in Colab).
- [ ] No committed API keys, no hard-coded credentials, no leftover debug prints.
- [ ] `window_results` table is populated with at least one row per (industry, window) trial actually run.
- [ ] Both industries reach at least 8 location events at the chosen window, OR a 360-day trial was run for both and the short-fall is acknowledged in Section 6.
- [ ] Cumulative window-tuning cost is at or below $3.00.
- [ ] Integrated map renders inline AND is exported to `maps/mp03_map_team_<NN>.html`.
- [ ] Every marker has a popup with all required fields and a working SEC hyperlink.
- [ ] Industry is visually distinguishable from event type on the map.
- [ ] Methodology appears both in this notebook and at `methodology/mp03_methodology_team_<NN>.md`.
- [ ] Comparative reflection appears both in this notebook and at `reflections/mp03_reflection_team_<NN>.md`.
- [ ] Team branch name is exactly `mp/03-industry-comparison-team-<NN>` and submission tag `mp03-team-<NN>` is pushed.
- [ ] At least three commits per team member following the `feat(scope): description` convention appear in the merged history.
- [ ] Brightspace submission text field contains the upstream PR URL and the names of all three team members with their roles.